# BTS — Etapa B1: LightGBM sobre las muestras completas, sin reconstruir Silver

Este cuaderno utiliza los Parquet finalizados en **Etapa A** (`4 000 000 / 1 000 000 / 1 000 000`). Conserva las variables, parámetros, horizonte temporal y evaluación de LightGBM del notebook original. Reduce el consumo de RAM **leyendo solo columnas necesarias** y convirtiendo numéricas a `float64` y categóricas a `category`.

**No vuelve a ejecutar la ingeniería histórica, ni hace un nuevo muestreo.** Guarda el modelo, probabilidades y etiquetas de validación/prueba para que las etapas B2–B4 y la comparación usen los mismos registros. Las métricas completas, bootstrap y experimentos de ablación se mantienen para etapas posteriores.

Ejecutar desde `SageMaker/src/`. Si el kernel se reinicia, los artefactos **confirmados** ya estarán guardados en disco; no asumas que una fase interrumpida terminó sin revisar los archivos.


**Variante experimental `float64`:** usa exactamente los Parquet de la etapa A, mismas muestras, columnas, niveles categóricos, semillas y parámetros del modelo. Solo cambia la conversión numérica `np.float32 → np.float64`, el nombre del checkpoint y su prefijo S3; los resultados originales `float32` permanecen intactos. Un cambio en el orden local original de la muestra puede seguir afectando las cifras, por lo que esta prueba aísla *solo el efecto de la conversión numérica en AWS*.


In [1]:
# Dependencias ya instaladas en el kernel que ejecutó B1/B3.
# Si falta alguna librería, instalar en el entorno del kernel antes de continuar.


In [2]:
from pathlib import Path
import os, gc, json, time, shutil
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss

ROOT = Path.cwd()
assert ROOT.name == "src", f"Abre el notebook desde SageMaker/src; actual={ROOT}"
PARTS_BASE = Path("../data/checkpoints/features_seed42_h240_chunk50000")
OUT_DIR = Path("../outputs/outputs_modeling/checkpoints/lightgbm_float64")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
TARGET = "DepDel15"
N_TRAIN, N_EVAL = 4_000_000, 1_000_000
COST_FP, COST_FN = 1, 3
LGB_MAX_TREES = 3000
LGB_EARLY_STOP = 100

print("Checkpoint:", PARTS_BASE.resolve())
print("Salida:", OUT_DIR.resolve())
print("Disco libre (GiB):", round(shutil.disk_usage(ROOT).free/1024**3, 2))

Checkpoint: /home/ec2-user/SageMaker/data/checkpoints/features_seed42_h240_chunk50000
Salida: /home/ec2-user/SageMaker/outputs/outputs_modeling/checkpoints/lightgbm_float64
Disco libre (GiB): 25.42


## 1. Validar checkpoints sin cargar filas en memoria

In [3]:
PARTS = {}
for split, expected in {"train": N_TRAIN, "val": N_EVAL, "test": N_EVAL}.items():
    folder = PARTS_BASE / split
    files = sorted(folder.glob("part_*.parquet"))
    assert (folder / "_SUCCESS").exists(), f"Falta _SUCCESS de {split}"
    assert files, f"No hay Parquet en {folder}"
    count = sum(pq.ParquetFile(p).metadata.num_rows for p in files)
    assert count == expected, f"{split}: {count:,} != {expected:,}"
    PARTS[split] = files
    print(f"OK {split}: {len(files)} Parquet, {count:,} filas")

OK train: 80 Parquet, 4,000,000 filas
OK val: 20 Parquet, 1,000,000 filas
OK test: 20 Parquet, 1,000,000 filas


## 2. Esquema de características y niveles de categóricas

Se extraen los niveles de las categorías **solo del entrenamiento**, como en el notebook original. No se vuelve a calcular ninguna variable.


In [4]:
CAT_COLS = ["Reporting_Airline", "Origin", "Dest", "route", "OriginState", "DestState", "DepTimeBlk"]
NUM_STATIC = ["Distance", "CRSElapsedTime", "dep_min", "arr_min", "Month", "DayOfWeek", "DayofMonth",
              "day_of_year", "is_weekend", "days_to_holiday"]
HIST_KEYS = ["origin", "route", "airline", "tail", "origin_hr"]
HIST_COLS = [f"{k}_rate_{suffix}" for k in HIST_KEYS for suffix in ("all", "30d")]
SCHED_COLS = ["origin_dep_load", "origin_day_load", "dest_arr_load", "tail_leg_num", "turnaround_min"]
STATUS_COLS = ["prev_leg_delayed", "tail_delayed_known", "tail_known_legs", "origin_known_today", "origin_delay_rate_today"]
FULL_FEATURES = NUM_STATIC + CAT_COLS + HIST_COLS + SCHED_COLS + STATUS_COLS
NUMERIC_COLS = [c for c in FULL_FEATURES if c not in CAT_COLS]
COLS_TO_READ = FULL_FEATURES + [TARGET]

schema_cols = set(pq.ParquetFile(PARTS['train'][0]).schema_arrow.names)
missing = [x for x in COLS_TO_READ if x not in schema_cols]
assert not missing, f"Faltan columnas: {missing}"

# Leer solo las columnas categóricas, un fragmento cada vez.
levels = {k: set() for k in CAT_COLS}
for i, path in enumerate(PARTS["train"], 1):
    table = pq.read_table(path, columns=CAT_COLS)
    df = table.to_pandas()
    for key in CAT_COLS:
        levels[key].update(df[key].dropna().unique().tolist())
    del df, table
    if i % 20 == 0: print("Niveles: fragmentos", i, "/", len(PARTS['train']))
CAT_LEVELS = {k: sorted(v) for k, v in levels.items()}
print({k: len(v) for k,v in CAT_LEVELS.items()})
(OUT_DIR / "feature_schema.json").write_text(json.dumps({"feature_names": FULL_FEATURES, "cat_cols": CAT_COLS,
    "cat_levels": CAT_LEVELS, "prediction_horizon_min": 240}, indent=2, ensure_ascii=False))
del levels; gc.collect()

Niveles: fragmentos 20 / 80
Niveles: fragmentos 40 / 80
Niveles: fragmentos 60 / 80
Niveles: fragmentos 80 / 80
{'Reporting_Airline': 17, 'Origin': 371, 'Dest': 372, 'route': 7268, 'OriginState': 53, 'DestState': 53, 'DepTimeBlk': 19}


33

## 3. Carga compacta solo de train y val

Aquí debe haber memoria suficiente para la matriz de LightGBM. Este enfoque reduce las copias de columnas que no intervienen en el modelo, pero **no garantiza** que el entrenamiento de 4 M quepa en 16 GiB. Si fallara aquí, hay que considerar datasets binarios LightGBM o procesamiento externo; no reducir automáticamente las muestras.


In [5]:
def load_compact(split):
    t0 = time.time()
    frames = []
    for i, path in enumerate(PARTS[split], 1):
        frame = pd.read_parquet(path, columns=COLS_TO_READ)
        for col in NUMERIC_COLS:
            frame[col] = pd.to_numeric(frame[col], errors="coerce").astype(np.float64)
        for col in CAT_COLS:
            frame[col] = pd.Categorical(frame[col], categories=CAT_LEVELS[col])
        frame[TARGET] = frame[TARGET].astype(np.uint8)
        frames.append(frame)
        if i % 20 == 0:
            print(f"  {split}: {i}/{len(PARTS[split])} fragmentos")
    result = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    print(f"{split}: {len(result):,} filas, {result.memory_usage(deep=True).sum()/1024**3:.2f} GiB, {time.time()-t0:.0f}s")
    return result

train = load_compact("train")
val = load_compact("val")
y_train = train.pop(TARGET).to_numpy(dtype=np.uint8, copy=False)
y_val = val.pop(TARGET).to_numpy(dtype=np.uint8, copy=False)
assert list(train.columns) == FULL_FEATURES
assert list(val.columns) == FULL_FEATURES
assert all(train[c].dtype == np.float64 for c in NUMERIC_COLS), "Train no es float64"
assert all(val[c].dtype == np.float64 for c in NUMERIC_COLS), "Val no es float64"
print("Prevalencia train:", round(float(y_train.mean()), 5), "val:", round(float(y_val.mean()),5))

  train: 20/80 fragmentos
  train: 40/80 fragmentos
  train: 60/80 fragmentos
  train: 80/80 fragmentos
train: 4,000,000 filas, 0.94 GiB, 8s
  val: 20/20 fragmentos
val: 1,000,000 filas, 0.23 GiB, 2s
Prevalencia train: 0.20898 val: 0.2227


## 4. Entrenar LightGBM original (sin bajar filas ni árboles)

In [6]:
LGB_PARAMS = dict(
    objective="binary", learning_rate=0.03, num_leaves=63, min_child_samples=2000,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=20.0,
    cat_smooth=100, min_data_per_group=500, max_cat_to_onehot=4,
    n_estimators=LGB_MAX_TREES, random_state=SEED, n_jobs=4, verbose=-1,
)

print("Entrenamiento de LightGBM con", len(train), "filas")
t0 = time.time()
model = lgb.LGBMClassifier(**LGB_PARAMS)
model.fit(
    train, y_train,
    eval_set=[(val, y_val)],
    eval_metric=["binary_logloss", "auc"],
    callbacks=[lgb.early_stopping(LGB_EARLY_STOP, first_metric_only=True, verbose=False)]
)
print("Árboles:", model.best_iteration_, "tiempo:", round(time.time()-t0, 1), "s")
model.booster_.save_model(str(OUT_DIR / "model.txt"))
p_val = model.predict_proba(val)[:,1]
np.savez_compressed(OUT_DIR / "val_predictions.npz", y=y_val, p=p_val)
print("PR-AUC validación:", average_precision_score(y_val, p_val))
(OUT_DIR / "run_config.json").write_text(json.dumps({
    "model": "LightGBM", "seed": SEED, "train_rows": len(train), "val_rows": len(val),
    "test_rows": N_EVAL, "numeric_dtype": "float64", "params": LGB_PARAMS, "best_iteration": int(model.best_iteration_),
    "train_end": "2024-12-31", "val_end": "2025-08-31", "pred_horizon_min": 240,
}, indent=2))
# Ya no necesitaremos la matriz de entrenamiento durante la evaluación de prueba.
del train, y_train
gc.collect()
print("Modelo y probabilidades de validación guardados en", OUT_DIR.resolve())

Entrenamiento de LightGBM con 4000000 filas


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Árboles: 930 tiempo: 716.6 s
PR-AUC validación: 0.5283471698943587
Modelo y probabilidades de validación guardados en /home/ec2-user/SageMaker/outputs/outputs_modeling/checkpoints/lightgbm_float64


## 5. Evaluación sobre prueba, sin recargar train

In [7]:
test = load_compact("test")
y_test = test.pop(TARGET).to_numpy(dtype=np.uint8, copy=False)
p_test = model.predict_proba(test)[:,1]
np.savez_compressed(OUT_DIR / "test_predictions.npz", y=y_test, p=p_test)
val_metrics = {
    "roc_auc": float(roc_auc_score(y_val, p_val)),
    "pr_auc": float(average_precision_score(y_val, p_val)),
    "brier": float(brier_score_loss(y_val, p_val)),
    "log_loss": float(log_loss(y_val, p_val)),
}
test_metrics = {
    "roc_auc": float(roc_auc_score(y_test, p_test)),
    "pr_auc": float(average_precision_score(y_test, p_test)),
    "brier": float(brier_score_loss(y_test, p_test)),
    "log_loss": float(log_loss(y_test, p_test)),
}
report = {"val": val_metrics, "test": test_metrics}
(OUT_DIR / "metrics.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
del test, model, y_test, y_val, p_test, p_val
gc.collect()
print("✔ Etapa B1 completada. Archivos:")
for f in sorted(OUT_DIR.iterdir()): print("  ",f.name, round(f.stat().st_size/1024**2,2), "MiB")

  test: 20/20 fragmentos
test: 1,000,000 filas, 0.23 GiB, 2s
{
  "val": {
    "roc_auc": 0.7524955991480693,
    "pr_auc": 0.5283471698943587,
    "brier": 0.14224413906256883,
    "log_loss": 0.4483486076447096
  },
  "test": {
    "roc_auc": 0.7592789941768864,
    "pr_auc": 0.5446309273825213,
    "brier": 0.14337794730035916,
    "log_loss": 0.45089214712124603
  }
}
✔ Etapa B1 completada. Archivos:
   feature_schema.json 0.13 MiB
   metrics.json 0.0 MiB
   model.txt 20.19 MiB
   run_config.json 0.0 MiB
   test_predictions.npz 7.34 MiB
   val_predictions.npz 7.34 MiB


## 6. Publicar checkpoint de modelo en S3 (opcional, recomendado)

Ejecutar **después** de completar la etapa B1. No se crea un endpoint todavía. Estos artefactos son resultados de comparación, no necesariamente el modelo ganador definitivo de tuning.


In [8]:
import boto3
BUCKET = "bts-flights-analysis"
PREFIX = "modeling/checkpoints/lightgbm-float64-original-seed42-h240"
s3 = boto3.Session().client("s3")
required = ["model.txt", "feature_schema.json", "run_config.json", "val_predictions.npz", "test_predictions.npz", "metrics.json"]
assert all((OUT_DIR / name).exists() for name in required), "Faltan artefactos: no subir una ejecución incompleta"
for name in required:
    s3.upload_file(str(OUT_DIR / name), BUCKET, f"{PREFIX}/{name}")
    print("Subido:", f"s3://{BUCKET}/{PREFIX}/{name}")
print("✔ Checkpoint de LightGBM respaldado en S3")

Subido: s3://bts-flights-analysis/modeling/checkpoints/lightgbm-float64-original-seed42-h240/model.txt
Subido: s3://bts-flights-analysis/modeling/checkpoints/lightgbm-float64-original-seed42-h240/feature_schema.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/lightgbm-float64-original-seed42-h240/run_config.json
Subido: s3://bts-flights-analysis/modeling/checkpoints/lightgbm-float64-original-seed42-h240/val_predictions.npz
Subido: s3://bts-flights-analysis/modeling/checkpoints/lightgbm-float64-original-seed42-h240/test_predictions.npz
Subido: s3://bts-flights-analysis/modeling/checkpoints/lightgbm-float64-original-seed42-h240/metrics.json
✔ Checkpoint de LightGBM respaldado en S3


## Siguientes etapas

- **B2:** regresión logística sobre 1 M de entrenamiento original.
- **B3:** XGBoost con 4 M de entrenamiento, en un kernel nuevo.
- **B4:** K-means con 1 M de entrenamiento, cuatro valores de K.
- **C:** experimentos de ablación/pesos de clase/deriva; guardar predicciones y métricas por modelo.
- **D:** comparar modelos con PR-AUC/ROC-AUC/Brier/ECE/costos, bootstrap pareado 100×300.000 y generar las figuras.
- **E:** tuning sobre el candidato seleccionado, Gold, S3 y SageMaker Endpoint.

El mismo orden de filas de `val`/`test` se obtiene leyendo **los mismos archivos** con orden de nombre `part_0000`, `part_0001`, ... No volver a hacer sample. Guardar y comparar siempre `y` con las predicciones.
